In [ ]:
from __future__ import annotations

from pathlib import Path
import os, sys
from dotenv import load_dotenv, find_dotenv

import json, time, argparse, hashlib, pathlib, re
import pymupdf as fitz  # keep for later steps (toc/chunking)
from langchain_text_splitters import SpacyTextSplitter, RecursiveCharacterTextSplitter

# ---------------- project root + env ----------------
def find_project_root() -> Path:
    p = Path.cwd()
    markers = {".git", "pyproject.toml", ".env"}
    for up in [p, *p.parents]:
        if any((up / m).exists() for m in markers):
            return up
    return p

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

env_path = find_dotenv(filename=".env", usecwd=True) or str(PROJECT_ROOT / ".env")
print("Loaded .env from:", env_path)
load_dotenv(env_path, override=False)

# ---------------- constants ----------------
from ingest.constants import SUPABASE_URL, SUPABASE_ANON_KEY, STORAGE_BUCKET, PDF_FILENAME, SUPABASE_SERVICE_ROLE_KEY
# from ingest.test.chunk import slug, norm_title, extract_text_pages, windows, flatten, descendants



In [ ]:
'''Supabase PDF Retrieval'''
from IPython.display import IFrame
from pathlib import Path
import sys, os
import hashlib, json
from ingest.chunk_import import fetch_pdf_from_storage, build_toc, chunk_sections
from ingest.constants import SUPABASE_URL, STORAGE_BUCKET, PDF_FILENAME, SUPABASE_SERVICE_ROLE_KEY

# Setup paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / "data" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Fetching PDF from private bucket...")
pdf_bytes = fetch_pdf_from_storage(
    SUPABASE_URL, 
    SUPABASE_SERVICE_ROLE_KEY,  # ← Use service role key for private buckets
    STORAGE_BUCKET, 
    PDF_FILENAME
)
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]

print(f"✅ Fetched PDF: {doc_key} ({len(pdf_bytes)} bytes)")

In [ ]:
'''Supabase PDF Retrieval'''

from IPython.display import IFrame
from pathlib import Path
import sys
import hashlib, json
from ingest.chunk_import import (
    fetch_pdf_from_storage, build_toc, chunk_sections
)
from ingest.embedding_import import generate_embeddings, validate_embeddings
from ingest.constants import SUPABASE_URL, SUPABASE_ANON_KEY, STORAGE_BUCKET, PDF_FILENAME

# Get project root (3 levels up from notebook)
NOTEBOOK_DIR = Path.cwd()  # /Users/matteogevi/Aurora-History-MVP/ingest/test/notebooks
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]  # /Users/matteogevi/Aurora-History-MVP

# Add to Python path if not already there
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python can now import from: ingest/")

OUT_DIR = Path("data/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fetch PDF
pdf_bytes = fetch_pdf_from_storage(SUPABASE_URL, SUPABASE_ANON_KEY, STORAGE_BUCKET, PDF_FILENAME)
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]

IFrame(src=pdf_url, width="100%", height=800)

In [ ]:
# Notebook: Ingest textbook
from pathlib import Path
import hashlib, json
from ingest.chunk import (
    fetch_pdf_from_storage, build_toc, chunk_sections
)
from ingest.embed import generate_embeddings, validate_embeddings
from constants import SUPABASE_URL, SUPABASE_ANON_KEY, STORAGE_BUCKET, PDF_FILENAME

OUT_DIR = Path("data/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fetch PDF
pdf_bytes = fetch_pdf_from_storage(SUPABASE_URL, SUPABASE_ANON_KEY, STORAGE_BUCKET, PDF_FILENAME)
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]

# 2. Build ToC
doc_key, toc, page_count = build_toc(pdf_bytes)
print(f"Built ToC: {page_count} pages")

# 3. Chunk sections
chunks = chunk_sections(pdf_bytes, toc, chunk_size=1000, overlap=150)
print(f"Generated {len(chunks)} chunks")

# 4. Generate embeddings
texts = [c["text"] for c in chunks]
embeddings = generate_embeddings(texts)

# 5. Validate
passed, issues = validate_embeddings(embeddings)
if not passed:
    print("⚠️ Embedding issues:", issues)
else:
    print("✅ All validations passed")

# 6. Add embeddings to chunks + doc_key
for i, c in enumerate(chunks):
    c["chunk_id"] = f"{doc_key}::{c['section_id']}::c{c['chunk_seq']:06d}"
    c["doc_key"] = doc_key
    c["embedding"] = embeddings[i].tolist()

# 7. Now push to Supabase (next cell)